In [1]:
# CELL 1: Setup paths and simple helpers

import pandas as pd
from pathlib import Path

# Base directories
BASE    = Path("/Users/davekokel/Projects/carp_v2")
IMPORT  = BASE / "seed_kits" / "legacy_import" / "working"
WORKING = BASE / "seed_kits" / "legacy_wrangling" / "working"

# v4 suffix + output helper
OUT_SUFFIX = "_v4"

def out_csv(name: str) -> Path:
    """
    Build a working CSV path with v4 suffix.
    Example: out_csv("imaging_roi_annotations_AUTO")
    -> WORKING / "imaging_roi_annotations_AUTO_v4.csv"
    """
    return WORKING / f"{name}{OUT_SUFFIX}.csv"

In [2]:
# CELL 2: Load canonical import flat markers

flat_import_path = IMPORT / "roi_all_markers_flat_complete.csv"  # adjust if name differs
df_flat = pd.read_csv(flat_import_path)

print("df_flat_import:", df_flat.shape)
df_flat.head()

df_flat_import: (217, 29)


,dataset_source,clutch_code,clutch_date,genotype_cross_label,genotype_base_codes,genotype_allele_codes,genotype_pretty,genotype_marker_fluor_codes,genotype_marker_tag_codes,treat_code,...,parent_female_filled,parent_male_filled,date_mount,mount_id,plate_id_filled,slot_id_filled,data_location_filled,roi_index,roi_name,data_path
0,mem_histone,IMG_CLT_20250130_01,2025-01-30,NaN,NaN,NaN,NaN,NaN,NaN,IMG_TRT_001,...,casper/rnf,casper/rnf,2025-01-31,NaN,SYN_20250131_plate1,SYN_20250131_plate1-slot1,U:\Data\20240911_Korra_Foundation\20250131_mem...,1,U:\Data\20240911_Korra_Foundation\20250131_mem...,U:\Data\20240911_Korra_Foundation\20250131_mem...
1,mem_histone,IMG_CLT_20250130_01,2025-01-30,NaN,NaN,NaN,NaN,NaN,NaN,IMG_TRT_001,...,casper/rnf,casper/rnf,2025-01-31,NaN,SYN_20250131_plate1,SYN_20250131_plate1-slot1,U:\Data\20240911_Korra_Foundation\20250131_mem...,1,U:\Data\20240911_Korra_Foundation\20250131_mem...,U:\Data\20240911_Korra_Foundation\20250131_mem...
2,mem_histone,IMG_CLT_20250203_02,2025-02-03,pDQM005_301,pDQM005,pDQM005:301,pDQM005(301),"tdmSG,tdmStayGold",NaN,IMG_TRT_004,...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),casper/rnf,2025-02-06,NaN,SYN_20250206_plate1,SYN_20250206_plate1-slot1,U:\Data\20240911_Korra_Foundation\20250204_mem...,1,U:\Data\20240911_Korra_Foundation\20250204_mem...,U:\Data\20240911_Korra_Foundation\20250204_mem...
3,mem_histone,IMG_CLT_20250203_02,2025-02-03,pDQM005_301,pDQM005,pDQM005:301,pDQM005(301),"tdmSG,tdmStayGold",NaN,IMG_TRT_004,...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),casper/rnf,2025-02-05,NaN,SYN_20250205_plate1,SYN_20250205_plate1-slot1,U:\Data\20240911_Korra_Foundation\20250204_mem...,1,U:\Data\20240911_Korra_Foundation\20250204_mem...,U:\Data\20240911_Korra_Foundation\20250204_mem...
4,mem_histone,IMG_CLT_20250203_02,2025-02-03,pDQM005_301,pDQM005,pDQM005:301,pDQM005(301),"tdmSG,tdmStayGold",NaN,IMG_TRT_004,...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),casper/rnf,2025-02-05,NaN,SYN_20250205_plate1,SYN_20250205_plate1-slot1,U:\Data\20240911_Korra_Foundation\20250204_mem...,1,U:\Data\20240911_Korra_Foundation\20250204_mem...,U:\Data\20240911_Korra_Foundation\20250204_mem...


In [4]:
# CELL 3: Collapse to one row per ROI with marker aggregation

def agg_uniq(series: pd.Series) -> str | None:
    vals = []
    for x in series:
        if pd.isna(x):
            continue
        s = str(x).strip()
        if not s:
            continue
        vals.append(s)
    if not vals:
        return None
    uniq = sorted(set(" ".join(s.split()) for s in vals))  # normalize whitespace
    return ",".join(uniq)

# Define ROI key
ROI_KEY = "data_path"  # adjust if needed

# Columns that should be aggregated as marker lists when repeated
marker_cols = [
    "genotype_marker_fluor_codes",
    "genotype_marker_tag_codes",
    "treatment_plasmid_base_codes",
    "treatment_rna_base_codes",
    "treatment_marker_fluor_codes",
    "treatment_marker_tag_codes",
    "all_marker_fluor_codes",
]

# Columns that should be taken from the first row (meta fields)
meta_cols = [
    # ROI_KEY is the group key; don't include it here
    "clutch_code",
    "clutch_date",
    "genotype_base_codes",
    "genotype_allele_codes",
    "genotype_pretty",
    "birthday_filled",
    "parent_female_filled",
    "parent_male_filled",
    "date_mount",
    "plate_id_filled",
    "slot_id_filled",
    "roi_index",
    "roi_name",
    "data_location_filled",
]

# Keep only columns that actually exist
meta_cols   = [c for c in meta_cols   if c in df_flat.columns]
marker_cols = [c for c in marker_cols if c in df_flat.columns]

grouped = df_flat.groupby(ROI_KEY)

# meta: one row per ROI, index = ROI_KEY
df_meta    = grouped[meta_cols].first().reset_index()   # ROI_KEY comes from index
df_markers = grouped[marker_cols].agg(agg_uniq).reset_index()

df_roi = pd.merge(df_meta, df_markers, on=ROI_KEY, how="left")

print("df_roi_v4:", df_roi.shape)
df_roi.head()

df_roi_v4: (207, 22)


,data_path,clutch_code,clutch_date,genotype_base_codes,genotype_allele_codes,genotype_pretty,birthday_filled,parent_female_filled,parent_male_filled,date_mount,...,roi_index,roi_name,data_location_filled,genotype_marker_fluor_codes,genotype_marker_tag_codes,treatment_plasmid_base_codes,treatment_rna_base_codes,treatment_marker_fluor_codes,treatment_marker_tag_codes,all_marker_fluor_codes
0,U:\Data\20240911_Korra_Foundation\20250131_mem...,IMG_CLT_20250130_01,2025-01-30,None,None,None,2025-01-30,casper/rnf,casper/rnf,2025-01-31,...,1,U:\Data\20240911_Korra_Foundation\20250131_mem...,U:\Data\20240911_Korra_Foundation\20250131_mem...,None,None,None,None,None,None,None
1,U:\Data\20240911_Korra_Foundation\20250131_mem...,IMG_CLT_20250130_01,2025-01-30,None,None,None,2025-01-30,casper/rnf,casper/rnf,2025-01-31,...,1,U:\Data\20240911_Korra_Foundation\20250131_mem...,U:\Data\20240911_Korra_Foundation\20250131_mem...,None,None,None,None,None,None,None
2,U:\Data\20240911_Korra_Foundation\20250204_mem...,IMG_CLT_20250203_02,2025-02-03,pDQM005,pDQM005:301,pDQM005(301),2025-02-03,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),casper/rnf,2025-02-04,...,1,U:\Data\20240911_Korra_Foundation\20250204_mem...,U:\Data\20240911_Korra_Foundation\20250204_mem...,"tdmSG,tdmStayGold",None,None,None,None,None,"tdmSG,tdmStayGold"
3,U:\Data\20240911_Korra_Foundation\20250204_mem...,IMG_CLT_20250203_02,2025-02-03,pDQM005,pDQM005:301,pDQM005(301),2025-02-03,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),casper/rnf,2025-02-04,...,1,U:\Data\20240911_Korra_Foundation\20250204_mem...,U:\Data\20240911_Korra_Foundation\20250204_mem...,"tdmSG,tdmStayGold",None,None,None,None,None,"tdmSG,tdmStayGold"
4,U:\Data\20240911_Korra_Foundation\20250204_mem...,IMG_CLT_20250203_02,2025-02-03,pDQM005,pDQM005:301,pDQM005(301),2025-02-03,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),casper/rnf,2025-02-05,...,1,U:\Data\20240911_Korra_Foundation\20250204_mem...,U:\Data\20240911_Korra_Foundation\20250204_mem...,"tdmSG,tdmStayGold",None,None,None,None,None,"tdmSG,tdmStayGold"


In [5]:
# CELL 4: Normalize column names for annotations

# Ensure roi_dir is present
if "roi_dir" in df_roi.columns:
    df_roi["roi_dir"] = df_roi["roi_dir"].astype(str)
else:
    df_roi["roi_dir"] = df_roi[ROI_KEY].astype(str)

# Normalize parents / birthday names
df_roi["parent_female"] = df_roi.get("parent_female_filled", df_roi.get("parent_female", None))
df_roi["parent_male"]   = df_roi.get("parent_male_filled", df_roi.get("parent_male", None))
df_roi["date_born"]     = df_roi.get("birthday_filled", df_roi.get("date_born", None))

# Normalize data_location / fish
df_roi["data_location"] = df_roi.get("data_location_filled", df_roi.get("data_location", None))
df_roi["fish"] = df_roi.get("fish", None)  # or derive from imaging sheet if needed

df_roi[[
    "roi_dir",
    "parent_female",
    "parent_male",
    "genotype_base_codes",
    "genotype_marker_fluor_codes",
    "genotype_marker_tag_codes",
    "treatment_marker_fluor_codes",
    "treatment_marker_tag_codes",
    "date_born",
    "plate_id_filled",
    "slot_id_filled",
    "roi_index",
    "roi_name",
]].head()

,roi_dir,parent_female,parent_male,genotype_base_codes,genotype_marker_fluor_codes,genotype_marker_tag_codes,treatment_marker_fluor_codes,treatment_marker_tag_codes,date_born,plate_id_filled,slot_id_filled,roi_index,roi_name
0,U:\Data\20240911_Korra_Foundation\20250131_mem...,casper/rnf,casper/rnf,None,None,None,None,None,2025-01-30,SYN_20250131_plate1,SYN_20250131_plate1-slot1,1,U:\Data\20240911_Korra_Foundation\20250131_mem...
1,U:\Data\20240911_Korra_Foundation\20250131_mem...,casper/rnf,casper/rnf,None,None,None,None,None,2025-01-30,SYN_20250131_plate1,SYN_20250131_plate1-slot1,1,U:\Data\20240911_Korra_Foundation\20250131_mem...
2,U:\Data\20240911_Korra_Foundation\20250204_mem...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),casper/rnf,pDQM005,"tdmSG,tdmStayGold",None,None,None,2025-02-03,SYN_20250204_plate1,SYN_20250204_plate1-slot1,1,U:\Data\20240911_Korra_Foundation\20250204_mem...
3,U:\Data\20240911_Korra_Foundation\20250204_mem...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),casper/rnf,pDQM005,"tdmSG,tdmStayGold",None,None,None,2025-02-03,SYN_20250204_plate1,SYN_20250204_plate1-slot1,1,U:\Data\20240911_Korra_Foundation\20250204_mem...
4,U:\Data\20240911_Korra_Foundation\20250204_mem...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),casper/rnf,pDQM005,"tdmSG,tdmStayGold",None,None,None,2025-02-03,SYN_20250205_plate1,SYN_20250205_plate1-slot1,1,U:\Data\20240911_Korra_Foundation\20250204_mem...


In [6]:
# CELL 5: Build roi_index_within_slot and roi_code from plate/slot/index

df_roi["roi_index_within_slot"] = df_roi.get("roi_index", None)

def make_roi_code(row):
    slot = str(row.get("slot_id_filled") or "").strip()
    idx  = row.get("roi_index_within_slot")
    if not slot or pd.isna(idx):
        return None
    try:
        i = int(idx)
    except Exception:
        return None
    return f"{slot}-roi{i:02d}"

df_roi["roi_code"] = df_roi.apply(make_roi_code, axis=1)

df_roi[[
    "roi_dir",
    "plate_id_filled",
    "slot_id_filled",
    "roi_index_within_slot",
    "roi_code",
]].head(20)

,roi_dir,plate_id_filled,slot_id_filled,roi_index_within_slot,roi_code
0,U:\Data\20240911_Korra_Foundation\20250131_mem...,SYN_20250131_plate1,SYN_20250131_plate1-slot1,1,SYN_20250131_plate1-slot1-roi01
1,U:\Data\20240911_Korra_Foundation\20250131_mem...,SYN_20250131_plate1,SYN_20250131_plate1-slot1,1,SYN_20250131_plate1-slot1-roi01
2,U:\Data\20240911_Korra_Foundation\20250204_mem...,SYN_20250204_plate1,SYN_20250204_plate1-slot1,1,SYN_20250204_plate1-slot1-roi01
3,U:\Data\20240911_Korra_Foundation\20250204_mem...,SYN_20250204_plate1,SYN_20250204_plate1-slot1,1,SYN_20250204_plate1-slot1-roi01
4,U:\Data\20240911_Korra_Foundation\20250204_mem...,SYN_20250205_plate1,SYN_20250205_plate1-slot1,1,SYN_20250205_plate1-slot1-roi01
5,U:\Data\20240911_Korra_Foundation\20250204_mem...,SYN_20250205_plate1,SYN_20250205_plate1-slot1,1,SYN_20250205_plate1-slot1-roi01
6,U:\Data\20240911_Korra_Foundation\20250204_mem...,SYN_20250205_plate1,SYN_20250205_plate1-slot1,1,SYN_20250205_plate1-slot1-roi01
7,U:\Data\20240911_Korra_Foundation\20250204_mem...,SYN_20250205_plate1,SYN_20250205_plate1-slot1,1,SYN_20250205_plate1-slot1-roi01
8,U:\Data\20240911_Korra_Foundation\20250204_mem...,SYN_20250206_plate1,SYN_20250206_plate1-slot1,1,SYN_20250206_plate1-slot1-roi01
9,U:\Data\20240911_Korra_Foundation\20250206_mem...,SYN_20250206_plate1,SYN_20250206_plate1-slot1,1,SYN_20250206_plate1-slot1-roi01


In [7]:
# CELL 6: Build final v4 annotations DataFrame

annot_cols = [
    "roi_dir",
    # core fields
    "parent_female", "parent_male",
    "genotype_pretty",
    "genotype_base_codes", "genotype_allele_codes",
    "genotype_marker_fluor_codes", "genotype_marker_tag_codes",
    "treatment_plasmid_base_codes", "treatment_rna_base_codes",
    "treatment_marker_fluor_codes", "treatment_marker_tag_codes",
    "all_marker_fluor_codes",
    # ROI metadata
    "date_born",
    "plate_id_filled", "slot_id_filled",
    "roi_index_within_slot", "roi_code",
    # optional extras that match your current schema as needed:
    # "data_location", "clutch_code", "clutch_date", etc.
]

# keep only columns that exist
annot_cols = [c for c in annot_cols if c in df_roi.columns]

annot_v4 = df_roi[annot_cols].copy()

print("annot_v4:", annot_v4.shape)
annot_v4.head()

annot_v4: (207, 18)


,roi_dir,parent_female,parent_male,genotype_pretty,genotype_base_codes,genotype_allele_codes,genotype_marker_fluor_codes,genotype_marker_tag_codes,treatment_plasmid_base_codes,treatment_rna_base_codes,treatment_marker_fluor_codes,treatment_marker_tag_codes,all_marker_fluor_codes,date_born,plate_id_filled,slot_id_filled,roi_index_within_slot,roi_code
0,U:\Data\20240911_Korra_Foundation\20250131_mem...,casper/rnf,casper/rnf,None,None,None,None,None,None,None,None,None,None,2025-01-30,SYN_20250131_plate1,SYN_20250131_plate1-slot1,1,SYN_20250131_plate1-slot1-roi01
1,U:\Data\20240911_Korra_Foundation\20250131_mem...,casper/rnf,casper/rnf,None,None,None,None,None,None,None,None,None,None,2025-01-30,SYN_20250131_plate1,SYN_20250131_plate1-slot1,1,SYN_20250131_plate1-slot1-roi01
2,U:\Data\20240911_Korra_Foundation\20250204_mem...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),casper/rnf,pDQM005(301),pDQM005,pDQM005:301,"tdmSG,tdmStayGold",None,None,None,None,None,"tdmSG,tdmStayGold",2025-02-03,SYN_20250204_plate1,SYN_20250204_plate1-slot1,1,SYN_20250204_plate1-slot1-roi01
3,U:\Data\20240911_Korra_Foundation\20250204_mem...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),casper/rnf,pDQM005(301),pDQM005,pDQM005:301,"tdmSG,tdmStayGold",None,None,None,None,None,"tdmSG,tdmStayGold",2025-02-03,SYN_20250204_plate1,SYN_20250204_plate1-slot1,1,SYN_20250204_plate1-slot1-roi01
4,U:\Data\20240911_Korra_Foundation\20250204_mem...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),casper/rnf,pDQM005(301),pDQM005,pDQM005:301,"tdmSG,tdmStayGold",None,None,None,None,None,"tdmSG,tdmStayGold",2025-02-03,SYN_20250205_plate1,SYN_20250205_plate1-slot1,1,SYN_20250205_plate1-slot1-roi01


In [8]:
# CELL 7: Write v4 annotations CSV

annot_path_v4 = out_csv("imaging_roi_annotations_AUTO")
annot_v4.to_csv(annot_path_v4, index=False)

annot_path_v4, annot_v4.shape

(PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/imaging_roi_annotations_AUTO_v4.csv'),
 (207, 18))

In [9]:
# CELL A: Load constructs and tags for fusion/localization mapping (v4)

from pathlib import Path
import pandas as pd

# paths reused from BASE
constructs_path = BASE / "seed_kits" / "legacy_wrangling" / "raw" / "constructs_plasmid.csv"
tags_path       = BASE / "seed_kits" / "legacy_wrangling" / "raw" / "tags.xlsx"

df_constructs = pd.read_csv(constructs_path)
df_tags       = pd.read_excel(tags_path)

print("df_constructs:", df_constructs.shape)
print("df_tags:", df_tags.shape)

df_constructs.head(), df_tags.head()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/raw/tags.xlsx'

In [ ]:
# CELL B: Build constructs mapping (plasmid_base_code -> fluor/tag/tag_pos)

cons_small = (
    df_constructs[["plasmid_code", "fluor_code", "tag_code", "tag_pos"]]
    .rename(columns={"plasmid_code": "plasmid_base_code"})
    .copy()
)
cons_small["plasmid_base_code"] = cons_small["plasmid_base_code"].astype(str).str.strip()

cons_small.head()

In [ ]:
# CELL C: Helpers for splitting code lists and building fusion labels

import pandas as pd

def split_codes(val: object) -> list[str]:
    if not isinstance(val, str) or not val.strip():
        return []
    return [c.strip() for c in val.split(",") if c.strip()]

def agg_uniq(series: pd.Series) -> str | None:
    vals = []
    for x in series:
        if pd.isna(x):
            continue
        s = str(x).strip()
        if not s:
            continue
        vals.append(s)
    if not vals:
        return None
    uniq = sorted(set(" ".join(s.split()) for s in vals))
    return ",".join(uniq)

def make_fusion_label(row: pd.Series) -> str | None:
    def norm(val: object) -> str:
        if pd.isna(val):
            return ""
        return str(val).strip()

    fluor = norm(row.get("fluor_code"))
    tag   = norm(row.get("tag_code"))
    pos   = norm(row.get("tag_pos"))

    if not fluor and not tag:
        return None
    if fluor and not tag:
        return fluor
    if not fluor and tag:
        return tag
    if fluor and tag and pos:
        return f"{fluor}::{tag}({pos})"
    return f"{fluor}::{tag}"

In [ ]:
# CELL D: Build genotype marker fusions from genotype_base_codes (v4)

gexp = (
    df_roi[["roi_dir", "genotype_base_codes"]]
    .dropna(subset=["genotype_base_codes"])
    .assign(code_list=lambda d: d["genotype_base_codes"].apply(split_codes))
    .explode("code_list")
    .rename(columns={"code_list": "plasmid_base_code"})
)

gexp["plasmid_base_code"] = gexp["plasmid_base_code"].astype(str).str.strip()

gjoin = gexp.merge(cons_small, how="left", on="plasmid_base_code")

gjoin["fusion_label"] = gjoin.apply(make_fusion_label, axis=1)

g_per_roi = (
    gjoin.groupby("roi_dir", as_index=False)
    .agg(genotype_marker_fusion_labels=("fusion_label", agg_uniq))
)

df_roi = df_roi.merge(g_per_roi, how="left", on="roi_dir")

df_roi[[
    "roi_dir",
    "genotype_base_codes",
    "genotype_marker_fluor_codes",
    "genotype_marker_tag_codes",
    "genotype_marker_fusion_labels",
]].head(20)

In [ ]:
# CELL E: Build treatment marker fusions from treatment base codes (v4)

def collect_all_treatment_codes(row: pd.Series) -> list[str]:
    codes: list[str] = []
    codes.extend(split_codes(row.get("treatment_plasmid_base_codes")))
    codes.extend(split_codes(row.get("treatment_rna_base_codes")))
    seen = set()
    out: list[str] = []
    for c in codes:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out

texp = (
    df_roi[["roi_dir", "treatment_plasmid_base_codes", "treatment_rna_base_codes"]]
    .assign(code_list=lambda d: d.apply(collect_all_treatment_codes, axis=1))
    .explode("code_list")
)

texp = texp[texp["code_list"].notna() & (texp["code_list"].astype(str).str.strip() != "")]
texp = texp.rename(columns={"code_list": "plasmid_base_code"})
texp["plasmid_base_code"] = texp["plasmid_base_code"].astype(str).str.strip()

tjoin = texp.merge(cons_small, how="left", on="plasmid_base_code")

tjoin["fusion_label"] = tjoin.apply(make_fusion_label, axis=1)

t_per_roi = (
    tjoin.groupby("roi_dir", as_index=False)
    .agg(treatment_marker_fusion_labels=("fusion_label", agg_uniq))
)

df_roi = df_roi.merge(t_per_roi, how="left", on="roi_dir")

df_roi[[
    "roi_dir",
    "treatment_plasmid_base_codes",
    "treatment_rna_base_codes",
    "treatment_marker_fluor_codes",
    "treatment_marker_tag_codes",
    "treatment_marker_fusion_labels",
]].head(20)

In [ ]:
# CELL F: Map tags to localizations and aggregate per ROI (v4)

# Normalize tags table to (tag_code, localization)
tags_map = (
    df_tags.rename(columns={"nickname": "tag_code"})[["tag_code", "localization"]]
    .dropna(subset=["tag_code"])
    .copy()
)
tags_map["tag_code"] = tags_map["tag_code"].astype(str).str.strip()

# Flatten genotype and treatment tag codes per ROI
rows = []

for _, row in df_roi.iterrows():
    roi = row["roi_dir"]
    for source, col in [
        ("genotype", "genotype_marker_tag_codes"),
        ("treatment", "treatment_marker_tag_codes"),
    ]:
        val = row.get(col)
        if val is None or pd.isna(val):
            continue
        for tag in split_codes(val):
            rows.append({"roi_dir": roi, "source": source, "tag_code": tag})

df_tags_exp = pd.DataFrame(rows)
if not df_tags_exp.empty:
    df_tags_exp = df_tags_exp.merge(tags_map, how="left", on="tag_code")

    # aggregate localizations per ROI/source
    loc_per_roi = (
        df_tags_exp.groupby(["roi_dir", "source"], as_index=False)
        .agg(localizations=("localization", agg_uniq))
    )

    # pivot into genotype vs treatment localization columns
    loc_wide = loc_per_roi.pivot(index="roi_dir", columns="source", values="localizations")
    loc_wide = loc_wide.rename(
        columns={
            "genotype": "genotype_marker_localizations",
            "treatment": "treatment_marker_localizations",
        }
    ).reset_index()

    df_roi = df_roi.merge(loc_wide, how="left", on="roi_dir")

df_roi[[
    "roi_dir",
    "genotype_marker_tag_codes",
    "genotype_marker_localizations",
    "treatment_marker_tag_codes",
    "treatment_marker_localizations",
]].head(20)

In [ ]:
# step g
annot_cols = [
    "roi_dir",
    "parent_female", "parent_male",
    "genotype_pretty",
    "genotype_base_codes", "genotype_allele_codes",
    "genotype_marker_fluor_codes", "genotype_marker_tag_codes",
    "genotype_marker_fusion_labels",
    "treatment_plasmid_base_codes", "treatment_rna_base_codes",
    "treatment_marker_fluor_codes", "treatment_marker_tag_codes",
    "treatment_marker_fusion_labels",
    "all_marker_fluor_codes",
    "genotype_marker_localizations",
    "treatment_marker_localizations",
    "date_born",
    "plate_id_filled", "slot_id_filled",
    "roi_index_within_slot", "roi_code",
]
annot_cols = [c for c in annot_cols if c in df_roi.columns]

annot_v4 = df_roi[annot_cols].copy()
annot_path_v4 = out_csv("imaging_roi_annotations_AUTO")
annot_v4.to_csv(annot_path_v4, index=False)

annot_path_v4, annot_v4.shape